# 第15次课：Pandas 分组聚合（模块三收官）

**地学编程基础** | 2学时（90分钟）

---

## 🎓 模块三即将完结

回顾模块三走过的 3 节课：

| 次数 | 主题 | 核心成果 |
|---|---|---|
| 第 13 次 | NumPy 基础 | 向量化运算、布尔索引 |
| 第 14 次 | Pandas 基础 | 读写 CSV、筛选、排序、添加列 |
| 第 15 次（今天）| Pandas 分组聚合 | groupby —— 数据分析三件套的最后一件 |

**今天结束后**——模块三全部完成——你的'**数据分析师入门**'技能就满级了！

---

## 📚 本节课学习目标

完成本节课后，你应该能够：

1. **理解** groupby 的核心思想（split-apply-combine）；
2. **掌握**单列分组+单聚合：`df.groupby('col')['val'].mean()`；
3. **掌握**多种聚合方法：mean/sum/max/min/count/std；
4. **掌握** `agg()` 同时做多种聚合；
5. **掌握**多列分组：`df.groupby(['year', 'month'])`；
6. **掌握** `reset_index()` 把分组结果变回普通 DataFrame；
7. **了解**简单透视表 `pivot_table`；
8. **能够**完成真实的'分组分析+导出'工作流。

---

## 📂 配套数据文件

本节课的目录结构：

```
lesson15/
├── Lesson15_Pandas分组聚合.ipynb     ← 你正在看的
└── data/
    └── weather_records.csv          ← 7 个观测站 × 12 个月的气象数据
```

**数据说明**：7 个观测站（分属华北/华东/华南/西南），每站 12 个月（2024年），共 84 条记录。

**字段**：station_id / city / region / year / month / temperature / rainfall

---

## 🤔 三件套的最后一件

**还记得第 14 课的'数据分析三件套'吗？**

- ✅ **筛选**：`df[df['列'] > 阈值]`
- ✅ **排序**：`df.sort_values('列')`
- ❓ **分组聚合**：???

**今天补齐最后一件 —— groupby**！

**典型问题**：
- 各**区域**的平均温度是多少？
- 各**月份**的总降水量是多少？
- 每个**观测站**的温度统计信息？
- 各**区域 × 月份**的双维度统计？

**这些都需要**：先按某个字段分组，再对每组做统计。

In [ ]:
# 先读取数据看一眼
import pandas as pd
import numpy as np

df = pd.read_csv("data/weather_records.csv")
print("形状：", df.shape)
df.head(10)

---

## 一、groupby 的核心思想

### 1.1 split-apply-combine 模式

**groupby 做的事**：

1. **Split（分组）**：按某列把数据拆分成多组
2. **Apply（应用）**：对每组分别做计算（如 mean、sum）
3. **Combine（合并）**：把结果合并成一张新表

**例子**：'各区域的平均温度'

```
原数据：                Split                    Apply                Combine
city  region  temp     →  华北组：[北京, 天津]  →  mean=14.5  →  华北 14.5
北京  华北    13.2                                                      华东 17.1
天津  华北    15.8         华东组：[上海, 杭州]  →  mean=17.1     华南 22.5
上海  华东    17.1                                                      西南 15.7
杭州  华东    17.0         华南组：[广州, 海口]  →  mean=22.5
广州  华南    22.5
海口  华南    24.5         西南组：[昆明]        →  mean=15.7
昆明  西南    15.7
```

**🌟 这就是 groupby 的本质**——'分组 → 应用 → 合并'三步走。

### 1.2 还记得第 7 次课的字典累加吗？

In [ ]:
# 第 7 次课的笨办法：用字典统计每个区域的城市数
data = [
    ("北京", "华北"),
    ("天津", "华北"),
    ("上海", "华东"),
    ("杭州", "华东"),
    ("广州", "华南"),
]

# 第 7 课的字典累加（约 5 行）
counts = {}
for city, region in data:
    counts[region] = counts.get(region, 0) + 1
print(counts)

In [ ]:
# 今天用 groupby（一行）
df_simple = pd.DataFrame(data, columns=["city", "region"])
df_simple.groupby("region").size()

**🌟 看到了吗？**

- **第 7 课**：5 行循环+字典累加
- **今天**：`df.groupby('region').size()` —— **一行**

**这就是 groupby 的力量**——**让分组统计变得无比简洁**。

---

## 二、单列分组 + 单聚合

### 2.1 最简单的形式

**🔑 语法**：

```python
df.groupby('分组列')['统计列'].聚合方法()
```

**例子**：各区域的平均温度

In [ ]:
# 各区域的平均温度
df.groupby("region")["temperature"].mean()

**关键观察**：
- `df.groupby('region')` —— 按 region 列分组（拆分）
- `['temperature']` —— 只看 temperature 这一列
- `.mean()` —— 对每组算平均
- 返回一个 **Series**：index 是组名（region 的值）、values 是每组的均值

### 2.2 各种聚合方法

In [ ]:
# 各区域温度的不同聚合
print("=== 各区域平均温度 ===")
print(df.groupby("region")["temperature"].mean())

print("\n=== 各区域最高温度 ===")
print(df.groupby("region")["temperature"].max())

print("\n=== 各区域温度标准差 ===")
print(df.groupby("region")["temperature"].std())

print("\n=== 各区域有多少条记录 ===")
print(df.groupby("region").size())

**📌 7 种最常用的聚合方法**：

| 方法 | 含义 |
|---|---|
| `.mean()` | 平均值 |
| `.sum()` | 求和 |
| `.max()` | 最大值 |
| `.min()` | 最小值 |
| `.median()` | 中位数 |
| `.std()` | 标准差 |
| `.count()` | 非空数量 |
| `.size()` | 总数量（含空）|

**和 `df['col'].mean()` 一样**——但**先分了组**。

### 2.3 按其他字段分组的例子

In [ ]:
# 各月份的全国平均温度（每月对全部观测站平均）
df.groupby("month")["temperature"].mean()

In [ ]:
# 各观测站的全年平均降水
df.groupby("city")["rainfall"].mean()

**🌟 结果是按字母（或拼音）排序的 Series**——可以直接做后续操作。

### 2.4 排序分组结果

In [ ]:
# 想看哪个区域温度最高？—— 分组后排序
df.groupby("region")["temperature"].mean().sort_values(ascending=False)

**🌟 链式调用**——`.groupby().mean().sort_values()` 一气呵成——这就是 Pandas 的优雅。

---

## 🎯 即时练习①

**练习目标**：熟悉单列分组 + 单聚合的标准模式。

**用 `df = pd.read_csv('data/weather_records.csv')` 完成**：

1. 计算**各区域**的平均**降水量**；
2. 计算**各观测站**的全年**最高温度**；
3. 计算**各月份**的**总降水量**（提示：sum）；
4. 找出**降水量最多**的区域（分组+sum+sort_values）；
5. 找出**温度最低**的月份（分组+mean+sort_values+head(1)）。

In [ ]:
# ===== 即时练习①.1：各区域平均降水量 =====




In [ ]:
# ===== 即时练习①.2：各观测站全年最高温度 =====




In [ ]:
# ===== 即时练习①.3：各月份总降水量 =====




In [ ]:
# ===== 即时练习①.4：降水量最多的区域 =====




In [ ]:
# ===== 即时练习①.5：温度最低的月份 =====




---

## 三、多种聚合 —— agg()

**有时候我们想'一次性看多个统计'**——比如同时看每组的均值、最大、最小。

### 3.1 一列多种聚合

In [ ]:
# 各区域温度的：均值、最高、最低、标准差
df.groupby("region")["temperature"].agg(["mean", "max", "min", "std"])

**🌟 关键观察**：
- `.agg([聚合方法名列表])` —— 一次得到多个统计
- 返回 DataFrame：行是组名、列是各种统计
- 列名就是聚合方法名（mean、max...）

### 3.2 多列同时聚合（不同列用不同方法）

In [ ]:
# 各区域：温度看均值，降水看总和
df.groupby("region").agg({
    "temperature": "mean",
    "rainfall": "sum"
})

**🔑 用字典指定**：
- key = 列名
- value = 聚合方法（字符串）

In [ ]:
# 进阶：每列也可以用多种方法
df.groupby("region").agg({
    "temperature": ["mean", "max"],
    "rainfall": ["sum", "mean"]
})

**📌 这就是 agg() 的两种用法**：
1. 一列多种聚合：`['mean', 'max', 'std']`（列表）
2. 多列不同聚合：`{'col1': 'mean', 'col2': 'sum'}`（字典）

### 3.3 一次拿到所有数值列的描述统计

In [ ]:
# 神技：分组后直接 describe
df.groupby("region")["temperature"].describe()

**🌟 `groupby().describe()`**——一行得到每组的 8 个统计量（count/mean/std/min/25%/50%/75%/max）——非常实用。

---

## 四、多列分组

**很多时候要按'多个维度'同时分组**——比如：'每个区域 × 每个月'的统计。

### 4.1 多列分组语法

**🔑 用列表传多个列名**：

In [ ]:
# 各区域 × 各月份的平均温度
result = df.groupby(["region", "month"])["temperature"].mean()
result.head(15)

**关键观察**：
- `groupby(['region', 'month'])` —— 列表传多个列名
- 返回的 Series 有'**多重索引**'（左侧两列：region、month）
- 每行对应一个'**区域+月份**'的组合

### 4.2 ⭐ reset_index() —— 把多重索引变回普通列

**多重索引看起来很复杂**——其实可以**一行变回普通 DataFrame**：

In [ ]:
# 用 reset_index 把分组列变回普通列
result_flat = df.groupby(["region", "month"])["temperature"].mean().reset_index()
result_flat.head(15)

**🌟 `.reset_index()`**：
- 把分组列从 index 移回成普通列
- 结果就是一个普通的 DataFrame
- 后续可以正常用 `df['region']`、`df.sort_values()` 等

**📌 经验法则**：**新手每次 groupby 后都加 `.reset_index()`**——避免多重索引带来的困扰。

### 4.3 多列分组+多列聚合

In [ ]:
# 各区域 × 各月份 —— 看温度均值 + 降水总和
result = df.groupby(["region", "month"]).agg({
    "temperature": "mean",
    "rainfall": "sum"
}).reset_index()

result.head(15)

---

## 🎯 即时练习②

**练习目标**：熟练运用多种聚合 + 多列分组。

**练习题**：

1. **每个区域**：温度的 mean、max、min（用 agg）；

2. **每个观测站**：温度看均值、降水看总和（用字典 agg）；

3. **各区域 × 各月份**：温度均值（多列分组+reset_index）；

4. 在第 3 题基础上，**筛选出温度高于 20°C 的'区域-月份'组合**。

In [ ]:
# ===== 即时练习②.1：每个区域多种聚合 =====




In [ ]:
# ===== 即时练习②.2：观测站多列不同聚合 =====




In [ ]:
# ===== 即时练习②.3：区域 × 月份分组 =====




In [ ]:
# ===== 即时练习②.4：在 3 的基础上筛选 > 20°C =====




---

## 五、简单透视表 pivot_table

**透视表（pivot table）就是 Excel 那个'**数据透视表**'**——把数据按两个维度展开成矩阵。

### 5.1 pivot_table 基础

In [ ]:
# 透视表：行是区域、列是月份、值是温度均值
pivot = df.pivot_table(
    index="region",       # 行
    columns="month",      # 列
    values="temperature", # 取值
    aggfunc="mean"        # 聚合方法
)
pivot

**🌟 看到了吗？**——这就是一个**典型的'区域 × 月份'矩阵**！

**4 个核心参数**：
- `index` —— 行维度
- `columns` —— 列维度
- `values` —— 要聚合的数值列
- `aggfunc` —— 聚合方法（默认 mean）

### 5.2 pivot_table vs groupby

**两者本质都在做'分组聚合'**——只是表现形式不同：

| 形式 | 输出 | 适合 |
|---|---|---|
| `groupby(['A','B'])` | **长表**（每个 A-B 组合一行）| 编程处理 |
| `pivot_table(index='A', columns='B')` | **宽表**（A 是行、B 是列）| 阅读、可视化 |

**📌 实战经验**：
- **写代码继续处理** → 用 groupby + reset_index
- **给老板看 / 做表 / 画热力图** → 用 pivot_table

In [ ]:
# 一个对比：同样的'区域 × 月份温度均值'

# 长表（groupby）
long_form = df.groupby(["region", "month"])["temperature"].mean().reset_index()
print("=== 长表（前 6 行） ===")
print(long_form.head(6))

# 宽表（pivot_table）
wide_form = df.pivot_table(index="region", columns="month", values="temperature", aggfunc="mean")
print("\n=== 宽表 ===")
print(wide_form)

**💡 数据是同样的——只是展示方式不同**。

---

## 🎯 即时练习③（综合实战）

**练习目标**：完成真实的'分组分析+导出'工作流。

**情境**：你是某气象局的数据分析员，要给领导出一份'**2024年区域气候报告**'——基于 weather_records.csv。

**任务**：

**步骤1**：读取数据；

**步骤2**：按 `region` 分组，计算每个区域的：
- 全年平均温度
- 全年总降水量
- 观测站数量（用 `nunique()` 统计 station_id 的不重复数）

**步骤3**：用 `agg()` 一次完成步骤2，存成 DataFrame；

**步骤4**：把结果按'**全年总降水量**'降序排列；

**步骤5**：写入 `output_region_summary.csv`（utf-8-sig）；

**步骤6**：用 `pivot_table` 生成'**区域 × 月份的温度热力表**'，写入 `output_region_month_temp.csv`；

**步骤7**：找出**'温度高于 25°C'的区域-月份组合**（多列分组+筛选）。

**完成后体会**：从原始 84 行数据 → 4 行的区域汇总 → 真实工作流的力量。

In [ ]:
# ===== 即时练习③ =====
import pandas as pd
import numpy as np

# 步骤1：读取




In [ ]:
# 步骤2-3：按 region 分组 + agg 多种聚合
# 提示：station_id 用 'nunique'




In [ ]:
# 步骤4：按总降水量降序




In [ ]:
# 步骤5：写入 output_region_summary.csv




In [ ]:
# 步骤6：pivot_table 区域×月份温度，写入 CSV




In [ ]:
# 步骤7：找出温度 > 25°C 的 region-month 组合




---

## 六、本节课小结

### 知识点回顾

| 知识点 | 关键语法 |
|---|---|
| split-apply-combine 思想 | 分组 → 应用 → 合并 |
| 单列分组+单聚合 | `df.groupby('col')['val'].mean()` |
| 各种聚合方法 | mean/sum/max/min/median/std/count/size |
| 一列多聚合 | `.agg(['mean', 'max', 'std'])` |
| 多列不同聚合 | `.agg({'col1':'mean', 'col2':'sum'})` |
| 多列分组 | `df.groupby(['A', 'B'])` |
| 拍平结果 | `.reset_index()` |
| 描述统计 | `.describe()` |
| 链式调用 | `.groupby().mean().sort_values()` |
| 透视表 | `pd.pivot_table(index, columns, values, aggfunc)` |

### 重点强调（重要程度从高到低）

1. ⭐⭐⭐ **`df.groupby('col')['val'].mean()`** —— 单列分组单聚合的标准模式
2. ⭐⭐⭐ **多列分组后加 `.reset_index()`** —— 避免多重索引
3. ⭐⭐⭐ **agg + 字典** —— 不同列做不同聚合
4. ⭐⭐ **链式调用** —— `.groupby().agg().sort_values()` 一气呵成
5. ⭐⭐ **pivot_table** —— 行列展开成宽表（适合阅读）
6. ⭐ **`nunique()`** —— 统计不重复值数量（实用）

### 课后作业

**1. 【基础】单列分组聚合（必做）**

用 `data/weather_records.csv` 完成：
1. 各**区域**的全年平均温度（按温度降序排）；
2. 各**月份**的总降水量；
3. 各**观测站**的最高温度和最低温度（用 agg）；
4. 找出**全年最热的观测站**（按温度均值排序后取第一个）。

**2. 【进阶】多列分组（必做）**

1. 各**区域 × 月份**的平均温度（reset_index）；
2. 找出哪个**'区域-月份'组合**降水量最大；
3. 用 pivot_table 生成'**区域 × 月份的降水热力表**'；
4. 把热力表写入 `output_rainfall_heatmap.csv`。

**3. 【挑战】真实分析报告（必做）**

生成一份'**各观测站全年统计报告**'：
- 列：station_id, city, region, mean_temp, max_temp, min_temp, total_rain
- 用 groupby + agg 实现
- 提示：分组键可以是多列 `['station_id', 'city', 'region']`
- 按 mean_temp 降序排
- 写入 `output_station_report.csv`

---

下节课预告：**第16次课 GeoPandas 基础（模块四开篇）** —— 真正的'地理信息系统'入门！我们会读 Shapefile、画地图、做空间查询——课程的核心目标。

---
---

## 🎓 模块三 完结撒花！

**模块三（科学计算）3 节课走过的路**：

| 课次 | 主题 | 你掌握的能力 |
|---|---|---|
| 13 | NumPy 基础 | 向量化运算、布尔索引、广播 |
| 14 | Pandas 基础 | DataFrame、筛选、排序、添加列 |
| 15 | Pandas 分组聚合 | groupby、agg、pivot_table |

**至此，你的能力已达'**数据分析师入门**'**：

✓ 处理大型数值数据（NumPy）

✓ 像 Excel 一样的表格分析（Pandas）

✓ 完整的'读 → 处理 → 分组分析 → 导出'工作流

✓ 应对真实数据的脏污（缺失值、类型转换）

**接下来 —— 模块四（地理空间数据处理）**：

- 第 16 次：GeoPandas 基础（矢量地理数据）
- 第 17 次：GeoPandas 空间分析（空间查询、缓冲区）
- 第 18 次：Rasterio（栅格遥感数据，NDVI 实战）
- 第 19 次：综合案例（一个完整的地学分析）

**模块四是这门课的核心目标**——15 节课打下的基础在这里全部派上用场！

---
---

# 📎 附录：参考答案

> **建议先独立完成所有练习题再查看答案。**

In [ ]:
# ===== 即时练习①.1 参考答案 =====
import pandas as pd
df = pd.read_csv("data/weather_records.csv")

print(df.groupby("region")["rainfall"].mean())

In [ ]:
# ===== 即时练习①.2 参考答案 =====
print(df.groupby("city")["temperature"].max())

In [ ]:
# ===== 即时练习①.3 参考答案 =====
print(df.groupby("month")["rainfall"].sum())

In [ ]:
# ===== 即时练习①.4 参考答案 =====
result = df.groupby("region")["rainfall"].sum().sort_values(ascending=False)
print(result)
print(f"\n降水量最多：{result.index[0]}（{result.iloc[0]:.1f} mm）")

In [ ]:
# ===== 即时练习①.5 参考答案 =====
result = df.groupby("month")["temperature"].mean().sort_values()
print("温度最低的月份：")
print(result.head(1))

In [ ]:
# ===== 即时练习②.1 参考答案 =====
df.groupby("region")["temperature"].agg(["mean", "max", "min"])

In [ ]:
# ===== 即时练习②.2 参考答案 =====
df.groupby("city").agg({
    "temperature": "mean",
    "rainfall": "sum"
})

In [ ]:
# ===== 即时练习②.3 参考答案 =====
result = df.groupby(["region", "month"])["temperature"].mean().reset_index()
print(result.head(15))

In [ ]:
# ===== 即时练习②.4 参考答案 =====
# 在 ②.3 结果基础上筛选
hot = result[result["temperature"] > 20]
print(f"温度 > 20°C 的区域-月份组合（{len(hot)} 个）：")
print(hot)

In [ ]:
# ===== 即时练习③ 综合实战参考答案 =====
import pandas as pd

# 步骤1：读取
df = pd.read_csv("data/weather_records.csv")

# 步骤2-3：按 region 分组 + agg
summary = df.groupby("region").agg({
    "temperature": "mean",
    "rainfall": "sum",
    "station_id": "nunique"
}).reset_index()

# 重命名列（更清晰）
summary.columns = ["region", "avg_temp", "total_rain", "station_count"]

# 步骤4：按总降水降序
summary = summary.sort_values("total_rain", ascending=False)
print("=== 区域汇总 ===")
print(summary)

In [ ]:
# 步骤5：写入 output_region_summary.csv
summary.to_csv("output_region_summary.csv", index=False, encoding="utf-8-sig")
print("output_region_summary.csv 已写入！")

In [ ]:
# 步骤6：pivot_table 区域×月份温度
pivot = df.pivot_table(
    index="region",
    columns="month",
    values="temperature",
    aggfunc="mean"
)
print("=== 区域 × 月份温度热力表 ===")
print(pivot.round(1))

pivot.to_csv("output_region_month_temp.csv", encoding="utf-8-sig")
print("\noutput_region_month_temp.csv 已写入！")

In [ ]:
# 步骤7：找温度 > 25°C 的 region-month 组合
all_combos = df.groupby(["region", "month"])["temperature"].mean().reset_index()
hot_combos = all_combos[all_combos["temperature"] > 25]
print(f"\n=== 温度 > 25°C 的区域-月份（共 {len(hot_combos)} 个）===")
print(hot_combos.sort_values("temperature", ascending=False))

---

## 课后作业参考答案

In [ ]:
# ===== 课后作业 1 参考答案 =====
import pandas as pd
df = pd.read_csv("data/weather_records.csv")

# 1. 区域平均温度（降序）
print("=== 各区域全年平均温度 ===")
print(df.groupby("region")["temperature"].mean().sort_values(ascending=False))

# 2. 各月份总降水量
print("\n=== 各月份总降水量 ===")
print(df.groupby("month")["rainfall"].sum())

# 3. 各观测站的 max 和 min
print("\n=== 各观测站温度极值 ===")
print(df.groupby("city")["temperature"].agg(["max", "min"]))

# 4. 全年最热的观测站
hottest = df.groupby("city")["temperature"].mean().sort_values(ascending=False).head(1)
print(f"\n=== 全年最热观测站 ===")
print(hottest)

In [ ]:
# ===== 课后作业 2 参考答案 =====
import pandas as pd
df = pd.read_csv("data/weather_records.csv")

# 1. 区域 × 月份平均温度
rm_temp = df.groupby(["region", "month"])["temperature"].mean().reset_index()
print("=== 区域 × 月份平均温度（前 10 行）===")
print(rm_temp.head(10))

# 2. 找'区域-月份'最大降水的组合
rm_rain = df.groupby(["region", "month"])["rainfall"].sum().reset_index()
max_combo = rm_rain.sort_values("rainfall", ascending=False).head(1)
print("\n=== 最大降水的区域-月份 ===")
print(max_combo)

# 3. 区域 × 月份降水热力表
rain_pivot = df.pivot_table(
    index="region",
    columns="month",
    values="rainfall",
    aggfunc="sum"
)
print("\n=== 降水热力表 ===")
print(rain_pivot.round(0))

# 4. 写入
rain_pivot.to_csv("output_rainfall_heatmap.csv", encoding="utf-8-sig")
print("\noutput_rainfall_heatmap.csv 已写入！")

In [ ]:
# ===== 课后作业 3 参考答案 =====
import pandas as pd
df = pd.read_csv("data/weather_records.csv")

# 多列分组+多种聚合
report = df.groupby(["station_id", "city", "region"]).agg({
    "temperature": ["mean", "max", "min"],
    "rainfall": "sum"
}).reset_index()

# 把多重列名拍平
report.columns = [
    "station_id", "city", "region",
    "mean_temp", "max_temp", "min_temp", "total_rain"
]

# 按 mean_temp 降序
report = report.sort_values("mean_temp", ascending=False)

print("=== 各观测站全年报告 ===")
print(report)

# 写入
report.to_csv("output_station_report.csv", index=False, encoding="utf-8-sig")
print("\noutput_station_report.csv 已写入！")